# Baseline Experiment — DeepSeek V3 + Qwen2.5-7B

Phase 1 first pass: 1 API model + 1 open-source model × 3 tasks = 6 experiments

---
## 0. Environment Setup

In [ ]:
import os
if not os.path.exists('/content/8307-project'):
    !git clone https://github.com/zigeng22/8307-project.git /content/8307-project
    print('Repo cloned')
else:
    !cd /content/8307-project && git pull
    print('Repo updated')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# verify datasets
import os
drive_data = '/content/drive/MyDrive/8307/Datasets'
assert os.path.exists(drive_data), f'Upload datasets to {drive_data} first'
print('OK:', os.listdir(drive_data))

In [ ]:
# symlink datasets
import os
link = '/content/Datasets'
src = '/content/drive/MyDrive/8307/Datasets'
if os.path.islink(link): os.remove(link)
os.symlink(src, link)
print(f'{link} -> {src}')

In [ ]:
!pip install -q transformers>=4.43.0 peft>=0.11.0 trl>=0.9.0 datasets accelerate
!pip install -q langchain langchain-community faiss-cpu sentence-transformers
!pip install -q rouge-score bert-score scikit-learn
!pip install -q openai anthropic tqdm pandas matplotlib seaborn

In [ ]:
import sys, os, torch
sys.path.insert(0, '/content/8307-project')
os.chdir('/content/8307-project')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('NO GPU — switch to A100')

In [ ]:
import os, getpass
os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste OpenRouter API Key: ')
# quick test with DeepSeek V3
from openai import OpenAI
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=os.environ['OPENROUTER_API_KEY'])
r = client.chat.completions.create(model='deepseek/deepseek-chat-v3-0324',
    messages=[{'role':'user','content':'Hi'}], max_tokens=5)
print('API OK:', r.choices[0].message.content)

In [ ]:
from data.loader import load_sentiment, load_mentalchat, load_medquad
from data.splitter import split_task1, split_task2, split_task3
print('Sentiment:', len(load_sentiment()))
print('MentalChat:', len(load_mentalchat()))
print('MedQuAD:', len(load_medquad()))
print('All data OK')

---
## 1. DeepSeek V3 — Baseline (API)

In [ ]:
!python experiments/run_baseline.py --model deepseek-v3 --task task1

In [ ]:
!python experiments/run_baseline.py --model deepseek-v3 --task task2

In [ ]:
!python experiments/run_baseline.py --model deepseek-v3 --task task3

---
## 2. Qwen2.5-7B — Baseline (local GPU)

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task task1

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task task2

In [ ]:
!python experiments/run_baseline.py --model qwen2.5-7b --task task3

---
## 3. View Results

In [ ]:
import json
from pathlib import Path
for d in sorted(Path('results/baseline').iterdir()):
    if d.is_dir():
        print(f'\n{d.name}:')
        for f in sorted(d.glob('*_metrics.json')):
            print(f'  {f.stem}: {json.loads(f.read_text())}')

In [ ]:
!mkdir -p /content/drive/MyDrive/8307/results_backup
!cp -r results/ /content/drive/MyDrive/8307/results_backup/
print('Results backed up to Drive')